In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

Make the actor

In [ ]:
def build_actor(state_dim, action_dim, hidden_layers=[64, 64], learning_rate=0.0003):
    inputs = Input(shape=(state_dim,))
    
    x = inputs
    for units in hidden_layers:
        x = Dense(units, activation='relu')(x)

    output = Dense(action_dim, activation='softmax')(x)  # Probability distribution over actions
    
    model = Model(inputs, output)
    model.compile(optimizer=Adam(learning_rate), loss='categorical_crossentropy')
    
    return model

Make the critic

In [ ]:
def build_critic(state_dim, hidden_layers=[64, 64], learning_rate=0.0003):
    inputs = Input(shape=(state_dim,))
    
    x = inputs
    for units in hidden_layers:
        x = Dense(units, activation='relu')(x)

    output = Dense(1, activation='linear')(x)  # Value function output
    
    model = Model(inputs, output)
    model.compile(optimizer=Adam(learning_rate), loss='mse')
    
    return model

Define PPO agent

In [ ]:
import numpy as np

class PPOAgent:
    def __init__(self, state_dim, action_dim, gamma=0.99, clip_ratio=0.2, lam=0.95, epochs=10):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.clip_ratio = clip_ratio
        self.lam = lam #lambda, typical choice is 0.95
        self.epochs = epochs
        
        self.actor = build_actor(state_dim, action_dim)
        self.critic = build_critic(state_dim)

    def get_action(self, state):
        """Select action using the Actor network (probabilistic policy)."""
        state = state.reshape(1, -1)
        probabilities = self.actor.predict(state, verbose=2)[0] #verbose=2 means it will print loss and accuracy every epoch
        action = np.random.choice(self.action_dim, p=probabilities)  # Sample action
        return action, probabilities[action]  # Return action and probability

    def compute_advantage(self, rewards, values):
        """Compute advantage estimates using GAE (Generalized Advantage Estimation)."""
        advantages = np.zeros_like(rewards)
        last_advantage = 0

        for t in reversed(range(len(rewards) - 1)):
            delta = rewards[t] + self.gamma * values[t + 1] - values[t]
            advantages[t] = last_advantage = delta + self.gamma * self.lam * last_advantage

        return advantages

Make a training function

In [ ]:
def train_ppo(env, agent, batch_size=64, episodes=1000):
    for episode in range(episodes):
        state, _ = env.reset()
        done = False
        states, actions, rewards, old_probs, values = [], [], [], [], []
        
        while not done:
            action, prob = agent.get_action(state)
            value = agent.critic.predict(state.reshape(1, -1), verbose=0)[0]

            next_state, reward, done, _, _ = env.step(action)

            states.append(state)
            actions.append(action)
            rewards.append(reward)
            old_probs.append(prob)
            values.append(value)

            state = next_state
        
        # Compute advantages
        values.append(0)  # Add dummy value for last step
        advantages = agent.compute_advantage(rewards, values)

        # Train Actor and Critic
        agent.actor.fit(np.array(states), np.array(actions), epochs=agent.epochs, verbose=0)
        agent.critic.fit(np.array(states), np.array(rewards), epochs=agent.epochs, verbose=0)

        print(f"Episode {episode+1}/{episodes}, Reward: {sum(rewards):.2f}")

env = DiceWarsEnv()
agent = PPOAgent(state_dim=10, action_dim=4)
train_ppo(env, agent)